# HW2 Part 1 — Data Cleaning and Validation

This notebook inspects `ohlc_hw`, records data-quality issues before repair, cleans only defensible cases, quarantines unresolved rows with reasons, and recreates `ohlc_hw_cleaned` in the local database. The source table is left unchanged.

In [1]:
from pathlib import Path
from html import unescape
import re
import sqlite3

import numpy as np
import pandas as pd

# Run this notebook with the personal HW2 folder as the working directory.
db_path = Path("data/data.db")
if not db_path.exists():
    raise FileNotFoundError(f"Database not found: {db_path.resolve()}")

with sqlite3.connect(db_path) as connection:
    raw = pd.read_sql_query("SELECT * FROM ohlc_hw", connection)

raw["ts"] = pd.to_datetime(raw["ts"], errors="coerce")
print("Database:", db_path.resolve())
print("Raw rows and columns:", raw.shape)
raw.info()
display(raw.describe(include="all").T)

column_meanings = pd.DataFrame({
    "column": ["ts", "open", "high", "low", "close", "adj_close", "volume", "ticker"],
    "meaning": ["Trading date", "Opening price", "Highest price", "Lowest price",
                "Closing price", "Split/dividend-adjusted close", "Trading volume", "Stock symbol"],
})
display(column_meanings)
print("Date range:", raw["ts"].min().date(), "to", raw["ts"].max().date())
print("Raw ticker labels:", raw["ticker"].nunique())
display(raw["ticker"].drop_duplicates().sort_values().to_frame().head(15))

Database: data/data.db
Raw rows and columns: (397711, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[ns]
 1   open       397711 non-null  float64       
 2   high       397711 non-null  float64       
 3   low        397711 non-null  float64       
 4   close      397711 non-null  float64       
 5   adj_close  377856 non-null  float64       
 6   volume     397711 non-null  int64         
 7   ticker     397711 non-null  object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 24.3+ MB


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
ts,397711,NaN,NaN,NaN,1998-04-17 21:18:45.292485888,1962-01-02 00:00:00,1985-05-28 00:00:00,2000-01-06 00:00:00,2012-08-21 00:00:00,2024-12-06 00:00:00,NaN
open,397711.0,NaN,NaN,NaN,46.285479,0.0,4.209938,19.85,57.53,604.26,67.132594
high,397711.0,NaN,NaN,NaN,46.74766,0.005208,4.24,20.06,58.13,608.63,67.77821
low,397711.0,NaN,NaN,NaN,45.816279,0.004801,4.16,19.63,56.95,597.88,66.477178
close,397711.0,NaN,NaN,NaN,46.292903,0.005208,4.2,19.86,57.53,605.4,67.141879
adj_close,377856.0,NaN,NaN,NaN,51.65225,0.002945,1.39,11.96,43.77,10442.510783,194.584045
volume,397711.0,NaN,NaN,NaN,30977733.689946,0.0,1730700.0,4648200.0,11023796.5,9276606399.0,126623650.175253
ticker,397711,150,PG,15450,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,column,meaning
0,ts,Trading date
1,open,Opening price
2,high,Highest price
3,low,Lowest price
4,close,Closing price
5,adj_close,Split/dividend-adjusted close
6,volume,Trading volume
7,ticker,Stock symbol


Date range: 1962-01-02 to 2024-12-06
Raw ticker labels: 150


,ticker
102,"<a target=""_blank""><span style=""color: blue"">A..."
5005,"<a target=""_blank""><span style=""color: blue"">A..."
1126,"<a target=""_blank""><span style=""color: blue"">A..."
1839,"<a target=""_blank""><span style=""color: blue"">A..."
2457,"<a target=""_blank""><span style=""color: blue"">B..."
1228,"<a target=""_blank""><span style=""color: blue"">C..."
1164,"<a target=""_blank""><span style=""color: blue"">C..."
9,"<a target=""_blank""><span style=""color: blue"">C..."
568,"<a target=""_blank""><span style=""color: blue"">C..."
141,"<a target=""_blank""><span style=""color: blue"">D..."


## Standardize labels and define reusable checks

Ticker text can contain HTML, extra spaces, or inconsistent case. Normalize it before grouping. The date-gap detector compares each stock's observed dates with the supplied reference calendar, so exchange holidays are not mistaken for missing trading days.

In [2]:
def normalize_ticker(value):
    "Remove HTML markup and standardize a ticker label."
    text = unescape(str(value))
    return re.sub(r"<[^>]*>", "", text).strip().upper()


def find_missing_dates(observed_dates, reference_dates):
    "Return reference-calendar dates absent from the observed series."
    observed = set(pd.to_datetime(pd.Series(observed_dates)).dt.normalize())
    reference = set(pd.to_datetime(pd.Series(reference_dates)).dt.normalize())
    return sorted(reference - observed)


def flag_three_sigma_returns(returns):
    "Flag both tails beyond three sample standard deviations from the mean."
    values = pd.to_numeric(returns, errors="coerce")
    mean = values.mean()
    std = values.std(ddof=1)
    if pd.isna(std) or std == 0:
        return pd.Series(False, index=values.index)
    return values.notna() & ((values - mean).abs() > 3 * std)


# Small examples demonstrate a missing-date detector and an injected return outlier.
demo_calendar = pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04"])
demo_observed = demo_calendar.delete(1)
print("Injected missing date detected:", find_missing_dates(demo_observed, demo_calendar))
demo_returns = pd.Series([0.001] * 30 + [0.20])
print("Injected 20% return flagged:", bool(flag_three_sigma_returns(demo_returns).iloc[-1]))

Injected missing date detected: [Timestamp('2024-01-03 00:00:00')]
Injected 20% return flagged: True


In [3]:
# Normalize ticker labels and remove exact duplicates after normalization.
work = raw.copy()
work["ticker_original"] = work["ticker"]
work["ticker"] = work["ticker_original"].map(normalize_ticker)

canonical_columns = [column for column in work.columns if column != "ticker_original"]
exact_duplicate_mask = work.duplicated(subset=canonical_columns, keep="first")
rows_before_dedup = len(work)
clean = work.loc[~exact_duplicate_mask].copy()

key_sizes = clean.groupby(["ticker", "ts"], dropna=False).size()
conflicting_keys = key_sizes[key_sizes > 1]
print("Rows before deduplication:", rows_before_dedup)
print("Exact duplicate rows removed:", int(exact_duplicate_mask.sum()))
print("Rows after deduplication:", len(clean))
print("Conflicting ticker-date keys:", len(conflicting_keys))
if not conflicting_keys.empty:
    display(clean.set_index(["ticker", "ts"]).loc[conflicting_keys.index].head(20))
    raise ValueError("Conflicting ticker-date records need review before cleaning.")

display(work[["ticker_original", "ticker"]].drop_duplicates().head(15))
print("Normalized tickers:", sorted(work["ticker"].dropna().unique()))

Rows before deduplication: 397711
Exact duplicate rows removed: 29460
Rows after deduplication: 368251
Conflicting ticker-date keys: 0


,ticker_original,ticker
0,CAT,CAT
1,CVX,CVX
3,HD,HD
4,AAPL,AAPL
5,TRV,TRV
6,JNJ,JNJ
8,DIS,DIS
9,"<a target=""_blank""><span style=""color: blue"">C...",CSCO
10,SHW,SHW
11,KO,KO


Normalized tickers: ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


## Diagnose issues before repair

The quality report keeps one row per issue, so one observation may appear several times. High severity means the row cannot be trusted as-is; medium marks missing data or statistical flags for review; low marks informational conditions such as zero volume or exact duplicates. Daily-return outliers are review flags, not automatic deletion instructions. For trading-day gaps, the `ohlc` table is the accepted per-ticker reference calendar.

In [4]:
issue_rows = []

def add_issue(mask, issue_type, description, severity, frame=clean):
    "Append one quality-report row for each matching observation."
    selected = frame.loc[pd.Series(mask, index=frame.index).fillna(False)]
    for _, row in selected.iterrows():
        issue_rows.append({
            "ticker": row.get("ticker", "UNKNOWN"),
            "issue_type": issue_type,
            "issue_date": row.get("ts"),
            "description": description,
            "severity": severity,
        })

price_columns = ["open", "high", "low", "close", "adj_close"]
raw_price_columns = ["open", "high", "low", "close"]

# Missing cells, including each missing adjusted close, are recorded before repair.
for column in clean.columns:
    missing_mask = clean[column].isna()
    if missing_mask.any():
        add_issue(missing_mask, "missing_value", f"Missing value in {column}.",
                  "medium" if column in price_columns else "low")

# Compare against the supplied reference calendar ticker by ticker.
with sqlite3.connect(db_path) as connection:
    reference_calendar = pd.read_sql_query("SELECT ts, ticker FROM ohlc", connection)
reference_calendar["ts"] = pd.to_datetime(reference_calendar["ts"], errors="coerce")
reference_calendar["ticker"] = reference_calendar["ticker"].map(normalize_ticker)
missing_date_records = []
for symbol, ref_group in reference_calendar.groupby("ticker"):
    observed_dates = clean.loc[clean["ticker"].eq(symbol), "ts"]
    for missing_date in find_missing_dates(observed_dates, ref_group["ts"]):
        missing_date_records.append({
            "ticker": symbol,
            "issue_type": "missing_trading_date",
            "issue_date": missing_date,
            "description": "Date appears in the supplied ticker reference calendar but not in ohlc_hw.",
            "severity": "medium",
        })
issue_rows.extend(missing_date_records)

# Compare existing adjusted closes with the same-date adjustment factor in ohlc.
with sqlite3.connect(db_path) as connection:
    reference_prices = pd.read_sql_query("SELECT ts, ticker, close, adj_close FROM ohlc", connection)
reference_prices["ts"] = pd.to_datetime(reference_prices["ts"], errors="coerce")
reference_prices["ticker"] = reference_prices["ticker"].map(normalize_ticker)
reference_prices = reference_prices.drop_duplicates(["ticker", "ts"])
reference_prices["reference_factor"] = reference_prices["adj_close"] / reference_prices["close"]
reference_prices["expected_adj_close"] = reference_prices["close"] * reference_prices["reference_factor"]
adjustment_check = clean.merge(
    reference_prices[["ticker", "ts", "expected_adj_close"]],
    on=["ticker", "ts"], how="left", validate="one_to_one",
)
adj_consistent = np.isclose(
    adjustment_check["adj_close"], adjustment_check["expected_adj_close"],
    rtol=1e-5, atol=1e-8, equal_nan=False,
)
adj_mismatch = adjustment_check["adj_close"].notna() & ~adj_consistent
add_issue(adj_mismatch, "adjusted_price_mismatch",
          "Existing adjusted close disagrees with the same-date reference adjustment factor; replaced using the matched reference factor.",
          "high", frame=adjustment_check)
print("Existing adjusted closes inconsistent with reference:", int(adj_mismatch.sum()))

# Inspect simple close-to-close returns by ticker; use sample standard deviation.
return_frame = clean.sort_values(["ticker", "ts"]).copy()
return_frame["simple_return"] = return_frame.groupby("ticker")["close"].pct_change(fill_method=None)
return_frame["three_sigma_outlier"] = return_frame.groupby("ticker")["simple_return"].transform(flag_three_sigma_returns)
add_issue(return_frame["three_sigma_outlier"], "three_sigma_return",
          "Absolute daily simple return deviation exceeds three within-ticker sample standard deviations; review, do not delete automatically.",
          "medium", frame=return_frame)

low_open = clean["low"] > clean["open"]
open_high = clean["open"] > clean["high"]
low_close = clean["low"] > clean["close"]
close_high = clean["close"] > clean["high"]
bad_ohlc = low_open | open_high | low_close | close_high
add_issue(bad_ohlc, "invalid_ohlc", "Expected low <= open <= high and low <= close <= high.", "high")

numeric_prices = clean[price_columns].apply(pd.to_numeric, errors="coerce")
nonfinite_price = numeric_prices.notna() & ~np.isfinite(numeric_prices)
nonfinite_rows = nonfinite_price.any(axis=1)
nonpositive_rows = (numeric_prices.le(0) & numeric_prices.notna()).any(axis=1)
add_issue(nonfinite_rows, "nonfinite_price", "At least one price is infinite or cannot be represented as a finite number.", "high")
add_issue(nonpositive_rows, "nonpositive_price", "At least one raw or adjusted price is zero or negative.", "high")
volume_numeric = pd.to_numeric(clean["volume"], errors="coerce")
nonfinite_volume = volume_numeric.notna() & ~np.isfinite(volume_numeric)
add_issue(nonfinite_volume, "nonfinite_volume", "Volume must be a finite number.", "high")
add_issue(clean["volume"].eq(0), "zero_volume", "Zero volume may indicate stale data; retained unless another rule quarantines the row.", "low")
add_issue(clean["volume"].lt(0), "negative_volume", "Volume cannot be negative.", "high")

# Record duplicate rows as an issue on their ticker-date, even though exact copies are removed.
duplicated_keys = work.loc[exact_duplicate_mask, ["ticker", "ts"]].drop_duplicates()
for row in duplicated_keys.itertuples(index=False):
    issue_rows.append({"ticker": row.ticker, "issue_type": "exact_duplicate",
                       "issue_date": row.ts,
                       "description": "Exact duplicate row removed after ticker normalization.",
                       "severity": "low"})

quality_report = pd.DataFrame(issue_rows, columns=["ticker", "issue_type", "issue_date", "description", "severity"])
if not quality_report.empty:
    quality_report = quality_report.sort_values(["severity", "ticker", "issue_date", "issue_type"], kind="stable")
print("Missing reference-calendar dates:", len(missing_date_records))
gap_summary = pd.DataFrame(
    [{"ticker": symbol, "missing_trading_days": sum(record["ticker"] == symbol for record in missing_date_records),
      "reference_start": group["ts"].min().date(), "reference_end": group["ts"].max().date()}
     for symbol, group in reference_calendar.groupby("ticker")]
)
display(gap_summary)
for gap in gap_summary.itertuples(index=False):
    print(f"Ticker {gap.ticker} is missing {gap.missing_trading_days} trading days between {gap.reference_start} and {gap.reference_end}.")
print("Three-sigma return flags:", int(return_frame["three_sigma_outlier"].sum()))

# Compare the largest flagged raw-price moves with reference-adjusted returns.
outlier_review = return_frame.merge(
    adjustment_check[["ticker", "ts", "expected_adj_close"]],
    on=["ticker", "ts"], how="left", validate="one_to_one",
).sort_values(["ticker", "ts"])
outlier_review["reference_adjusted_return"] = outlier_review.groupby("ticker")["expected_adj_close"].pct_change(fill_method=None)
outlier_review["absolute_simple_return"] = outlier_review["simple_return"].abs()
review_columns = ["ticker", "ts", "simple_return", "reference_adjusted_return", "three_sigma_outlier"]
print("Largest three-sigma flags for manual review; these are retained unless repair is defensible:")
display(outlier_review.loc[outlier_review["three_sigma_outlier"]].nlargest(20, "absolute_simple_return")[review_columns])

print("Quality-report rows:", len(quality_report))
display(quality_report.groupby(["issue_type", "severity"]).size().rename("count").to_frame())
display(quality_report.head(20))

Existing adjusted closes inconsistent with reference: 10417
Missing reference-calendar dates: 0


,ticker,missing_trading_days,reference_start,reference_end
0,AAPL,0,1980-12-12,2024-10-18
1,AMGN,0,1983-06-17,2024-10-18
2,AMZN,0,1997-05-15,2024-12-06
3,AXP,0,1972-04-03,2024-10-18
4,BA,0,1962-01-02,2024-10-18
5,CAT,0,1962-01-02,2024-10-18
6,CRM,0,2004-06-23,2024-10-18
7,CSCO,0,1990-02-16,2024-10-18
8,CVX,0,1962-01-02,2024-12-06
9,DIS,0,1962-01-02,2024-12-06


Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18.
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18.
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06.
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18.
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18.
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18.
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18.
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18.
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06.
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06.
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18.
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18.
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18.
Ticker IBM is missing 0 trading days between 1962-01-02 and 202

Largest three-sigma flags for manual review; these are retained unless repair is defensible:


,ticker,ts,simple_return,reference_adjusted_return,three_sigma_outlier
318332,TRV,1976-10-28,1.000000,2.273973,True
319538,TRV,1981-08-10,1.000000,1.675159,True
318873,TRV,1978-12-20,0.997512,2.088973,True
213788,MCD,1968-01-02,-0.753620,-0.755881,True
4999,AAPL,2000-09-29,-0.518695,-0.518686,True
319539,TRV,1981-08-11,-0.500000,-0.626190,True
318333,TRV,1976-10-29,-0.491323,-0.689386,True
318874,TRV,1978-12-21,-0.490660,-0.669968,True
280845,NVDA,2000-03-07,0.424106,0.424226,True
281954,NVDA,2004-08-06,-0.352314,-0.352358,True


Quality-report rows: 63392


,,count
issue_type,severity,
adjusted_price_mismatch,high,10417
exact_duplicate,low,29460
invalid_ohlc,high,5
missing_value,medium,18374
nonpositive_price,high,5
three_sigma_return,medium,5056
zero_volume,low,75


,ticker,issue_type,issue_date,description,severity
21946,AAPL,adjusted_price_mismatch,1981-02-03,Existing adjusted close disagrees with the sam...,high
18815,AAPL,adjusted_price_mismatch,1981-04-28,Existing adjusted close disagrees with the sam...,high
25749,AAPL,adjusted_price_mismatch,1981-05-08,Existing adjusted close disagrees with the sam...,high
20434,AAPL,adjusted_price_mismatch,1981-07-29,Existing adjusted close disagrees with the sam...,high
22607,AAPL,adjusted_price_mismatch,1981-08-06,Existing adjusted close disagrees with the sam...,high
23675,AAPL,adjusted_price_mismatch,1981-08-13,Existing adjusted close disagrees with the sam...,high
19023,AAPL,adjusted_price_mismatch,1981-08-14,Existing adjusted close disagrees with the sam...,high
25695,AAPL,adjusted_price_mismatch,1981-10-08,Existing adjusted close disagrees with the sam...,high
24319,AAPL,adjusted_price_mismatch,1981-12-31,Existing adjusted close disagrees with the sam...,high
20237,AAPL,adjusted_price_mismatch,1982-01-15,Existing adjusted close disagrees with the sam...,high


## Demonstrate repair behavior on injected missing values

The real data has no missing raw OHLC or volume cells, so this small example demonstrates the required policies without changing the real observations: forward-fill an internal missing price within one ticker, fill missing volume with zero, and keep explicit repair flags. A leading gap remains unresolved and should be quarantined rather than extrapolated.

In [5]:
demo = pd.DataFrame({
    "ticker": ["DEMO"] * 3,
    "ts": pd.to_datetime(["2024-01-02", "2024-01-03", "2024-01-04"]),
    "open": [10.0, np.nan, 11.0],
    "high": [10.5, 11.5, 11.5],
    "low": [9.5, 10.0, 10.5],
    "close": [10.2, 11.0, 11.2],
    "volume": [100, np.nan, 130],
}).sort_values(["ticker", "ts"])
for column in raw_price_columns:
    demo[f"{column}_repaired"] = demo[column].isna()
    demo[column] = demo.groupby("ticker")[column].ffill()
demo["volume_repaired"] = demo["volume"].isna()
demo["volume"] = demo["volume"].fillna(0)
display(demo)

leading_gap = pd.Series([np.nan, 10.0]).ffill()
print("Leading gap stays missing after forward-fill:", bool(leading_gap.isna().iloc[0]))

,ticker,ts,open,high,low,close,volume,open_repaired,high_repaired,low_repaired,close_repaired,volume_repaired
0,DEMO,2024-01-02,10.0,10.5,9.5,10.2,100.0,False,False,False,False,False
1,DEMO,2024-01-03,10.0,11.5,10.0,11.0,0.0,True,False,False,False,True
2,DEMO,2024-01-04,11.0,11.5,10.5,11.2,130.0,False,False,False,False,False


Leading gap stays missing after forward-fill: True


## Repair defensible cases, quarantine unresolved rows, and save

Missing `adj_close` is reconstructed from the same ticker/date in `ohlc`: `adjusted close / reference close` gives that date's adjustment factor. Raw invalid prices are not guessed. The five zero-open observations are quarantined with an explicit reason because the reference table also has zero open for those same rows. Zero volume by itself is retained and flagged.

In [6]:
# Prepare clean, repair flags, and the adjustment reference.
prepared = clean.sort_values(["ticker", "ts"]).copy()
for column in raw_price_columns:
    prepared[f"{column}_repaired"] = prepared[column].isna()
prepared["volume_repaired"] = prepared["volume"].isna()
prepared["adj_close_repaired"] = False
prepared["three_sigma_return_flag"] = return_frame["three_sigma_outlier"].reindex(prepared.index).fillna(False).astype(bool)

# Drop rows with every raw OHLC value missing; retain them in the quarantine log.
all_raw_prices_missing = prepared[raw_price_columns].isna().all(axis=1)
quarantine_parts = []
if all_raw_prices_missing.any():
    dropped = prepared.loc[all_raw_prices_missing].copy()
    dropped["quarantine_reason"] = "All raw OHLC prices are missing; row removed before filling."
    quarantine_parts.append(dropped)
prepared = prepared.loc[~all_raw_prices_missing].copy()

# Fill only internal/previously observed raw prices within the same ticker.
for column in raw_price_columns:
    prepared[column] = prepared.groupby("ticker")[column].ffill()
prepared["volume"] = prepared["volume"].fillna(0)

reference = pd.read_sql_query("SELECT ts, ticker, close, adj_close FROM ohlc", sqlite3.connect(db_path))
reference["ts"] = pd.to_datetime(reference["ts"], errors="coerce")
reference["ticker"] = reference["ticker"].map(normalize_ticker)
reference = reference.drop_duplicates(["ticker", "ts"])
reference["adjustment_factor"] = reference["adj_close"] / reference["close"]
reference["expected_adj_close"] = reference["close"] * reference["adjustment_factor"]
reference = reference[["ticker", "ts", "close", "adjustment_factor", "expected_adj_close"]].rename(columns={"close": "reference_close"})
prepared = prepared.merge(reference, on=["ticker", "ts"], how="left", validate="one_to_one")

valid_factor = (
    prepared["reference_close"].gt(0)
    & np.isfinite(prepared["adjustment_factor"])
    & prepared["adjustment_factor"].gt(0)
    & prepared["close"].gt(0)
)
expected_adjusted = prepared["close"] * prepared["adjustment_factor"]
adjusted_matches_reference = np.isclose(
    prepared["adj_close"], expected_adjusted, rtol=1e-5, atol=1e-8, equal_nan=False
)
repair_adjusted = valid_factor & (~adjusted_matches_reference)
prepared.loc[repair_adjusted, "adj_close"] = expected_adjusted.loc[repair_adjusted]
prepared.loc[repair_adjusted, "adj_close_repaired"] = True
can_reconstruct = repair_adjusted

# Mark every unresolved row with one or more concrete reasons.
reason_masks = {
    "unfilled_raw_price": prepared[raw_price_columns].isna().any(axis=1),
    "missing_adjusted_close": prepared["adj_close"].isna(),
    "invalid_ohlc_bounds": (
        (prepared["low"] > prepared["open"]) | (prepared["open"] > prepared["high"])
        | (prepared["low"] > prepared["close"]) | (prepared["close"] > prepared["high"])
    ),
    "nonpositive_or_nonfinite_price": (
        ~np.isfinite(prepared[price_columns].apply(pd.to_numeric, errors="coerce"))
    ).any(axis=1) | prepared[price_columns].le(0).any(axis=1),
    "negative_or_nonfinite_volume": (~np.isfinite(prepared["volume"])) | prepared["volume"].lt(0),
}
invalid = pd.Series(False, index=prepared.index)
for mask in reason_masks.values():
    invalid |= mask.fillna(True)

if invalid.any():
    unresolved = prepared.loc[invalid].copy()
    unresolved["quarantine_reason"] = [
        "; ".join(name for name, mask in reason_masks.items() if bool(mask.loc[index]))
        for index in unresolved.index
    ]
    quarantine_parts.append(unresolved)

quarantine = pd.concat(quarantine_parts, ignore_index=True) if quarantine_parts else pd.DataFrame()
cleaned = prepared.loc[~invalid].drop(columns=["reference_close", "adjustment_factor"]).copy()

# Serialize dates as ISO calendar dates for SQL date filtering.
cleaned["ts"] = pd.to_datetime(cleaned["ts"]).dt.strftime("%Y-%m-%d")
if not quarantine.empty:
    quarantine["ts"] = pd.to_datetime(quarantine["ts"]).dt.strftime("%Y-%m-%d")

with sqlite3.connect(db_path) as connection:
    cleaned.to_sql("ohlc_hw_cleaned", connection, if_exists="replace", index=False)
    quarantine.to_sql("ohlc_hw_quarantine", connection, if_exists="replace", index=False)

print("Rows before deduplication:", rows_before_dedup)
print("Exact duplicate rows removed:", int(exact_duplicate_mask.sum()))
print("Rows retained:", len(cleaned))
print("Rows quarantined:", len(quarantine))
print("Adjusted closes filled or corrected from matched reference:", int(can_reconstruct.sum()))
print("Rows with repaired raw prices:", int(cleaned[[f"{c}_repaired" for c in raw_price_columns]].any(axis=1).sum()))
print("Rows with filled volume:", int(cleaned["volume_repaired"].sum()))
print("Rows with zero volume retained:", int(cleaned["volume"].eq(0).sum()))
display(quarantine[["ts", "ticker", "ticker_original", "open", "volume", "quarantine_reason"]].head(20) if not quarantine.empty else quarantine)

Rows before deduplication: 397711
Exact duplicate rows removed: 29460
Rows retained: 368246
Rows quarantined: 5
Adjusted closes filled or corrected from matched reference: 28791
Rows with repaired raw prices: 0
Rows with filled volume: 0
Rows with zero volume retained: 74


,ts,ticker,ticker_original,open,volume,quarantine_reason
0,1962-01-02,CVX,CVX,0.0,105840,invalid_ohlc_bounds; nonpositive_or_nonfinite_...
1,1962-01-02,JNJ,JNJ,0.0,0,invalid_ohlc_bounds; nonpositive_or_nonfinite_...
2,1966-07-05,MCD,MCD,0.0,388800,invalid_ohlc_bounds; nonpositive_or_nonfinite_...
3,1962-01-02,MRK,MRK,0.0,633830,invalid_ohlc_bounds; nonpositive_or_nonfinite_...
4,1962-01-02,PG,PG,0.0,192000,invalid_ohlc_bounds; nonpositive_or_nonfinite_...


## Validate the saved tables

The checks below read the result back from SQLite, rather than relying only on the in-memory dataframe. The original `ohlc_hw` table remains available for reproducibility.

In [7]:
with sqlite3.connect(db_path) as connection:
    saved = pd.read_sql_query("SELECT * FROM ohlc_hw_cleaned", connection)
    quarantine_saved = pd.read_sql_query("SELECT * FROM ohlc_hw_quarantine", connection)

saved["ts"] = pd.to_datetime(saved["ts"], errors="coerce")
saved_prices = saved[price_columns].apply(pd.to_numeric, errors="coerce")
saved_ohlc_valid = (
    saved["low"].le(saved["open"]) & saved["open"].le(saved["high"])
    & saved["low"].le(saved["close"]) & saved["close"].le(saved["high"])
)
with sqlite3.connect(db_path) as connection:
    saved_reference = pd.read_sql_query("SELECT ts, ticker, close, adj_close FROM ohlc", connection)
saved_reference["ts"] = pd.to_datetime(saved_reference["ts"], errors="coerce")
saved_reference["ticker"] = saved_reference["ticker"].map(normalize_ticker)
saved_reference["reference_factor"] = saved_reference["adj_close"] / saved_reference["close"]
saved_check = saved.merge(
    saved_reference[["ticker", "ts", "reference_factor"]],
    on=["ticker", "ts"], how="left", validate="one_to_one",
)
expected_saved_adj = saved_check["close"] * saved_check["reference_factor"]
adj_consistent_after = np.isclose(saved_check["adj_close"], expected_saved_adj, rtol=1e-5, atol=1e-8)
validation = pd.Series({
    "Rows in cleaned table": len(saved),
    "Rows in quarantine table": len(quarantine_saved),
    "Duplicate ticker-date keys": saved.duplicated(["ticker", "ts"]).sum(),
    "Missing values in cleaned table": int(saved.isna().sum().sum()),
    "Non-finite or nonpositive price cells": int((~np.isfinite(saved_prices) | saved_prices.le(0)).sum().sum()),
    "Invalid OHLC rows": int((~saved_ohlc_valid).sum()),
    "Adjusted-close rows inconsistent with reference": int((~adj_consistent_after).sum()),
    "Non-finite or negative volume rows": int((~np.isfinite(saved["volume"]) | saved["volume"].lt(0)).sum()),
    "Rows with repair flags": int(saved[["open_repaired", "high_repaired", "low_repaired", "close_repaired", "volume_repaired", "adj_close_repaired"]].any(axis=1).sum()),
})
display(validation.to_frame("count"))

price_volume_columns = ["open", "high", "low", "close", "adj_close", "volume"]
before_after_stats = pd.concat({
    "before (deduplicated raw)": clean[price_volume_columns].describe().loc[["count", "mean", "std", "min", "max"]].T,
    "after (retained cleaned)": saved[price_volume_columns].describe().loc[["count", "mean", "std", "min", "max"]].T,
}, axis=1)
display(before_after_stats)
assert validation["Duplicate ticker-date keys"] == 0
assert validation["Missing values in cleaned table"] == 0
assert validation["Non-finite or nonpositive price cells"] == 0
assert validation["Invalid OHLC rows"] == 0
assert validation["Adjusted-close rows inconsistent with reference"] == 0
assert validation["Non-finite or negative volume rows"] == 0

,count
Rows in cleaned table,368246
Rows in quarantine table,5
Duplicate ticker-date keys,0
Missing values in cleaned table,0
Non-finite or nonpositive price cells,0
Invalid OHLC rows,0
Adjusted-close rows inconsistent with reference,0
Non-finite or negative volume rows,0
Rows with repair flags,28791


before (deduplicated raw)                                        \
                              count          mean           std       min   
open                       368251.0  4.630877e+01  6.714301e+01  0.000000   
high                       368251.0  4.677120e+01  6.778907e+01  0.005208   
low                        368251.0  4.583977e+01  6.648920e+01  0.004801   
close                      368251.0  4.631671e+01  6.715469e+01  0.005208   
adj_close                  349877.0  5.171154e+01  1.950017e+02  0.002945   
volume                     368251.0  3.095142e+07  1.266145e+08  0.000000   

                        after (retained cleaned)                              \
                    max                    count          mean           std   
open       6.042600e+02                 368246.0  4.630939e+01  6.714325e+01   
high       6.086300e+02                 368246.0  4.677182e+01  6.778932e+01   
low        5.978800e+02                 368246.0  4.584038e+01  6.648945e+01   
close      6.054000e+02                 368246.0  4.631733e+01  6.715494e+01   
adj_close  1.044251e+04                 368246.0  3.727467e+01  6.358787e+01   
volume     9.276606e+09                 368246.0  3.095184e+07  1.266153e+08   

                                   
                min           max  
open       0.005208  6.042600e+02  
high       0.005208  6.086300e+02  
low        0.004801  5.978800e+02  
close      0.005208  6.054000e+02  
adj_close  0.002945  6.054000e+02  
volume     0.000000  9.276606e+09